# Model Comparison

**Topic:** Supervised Learning — Capstone

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from ipywidgets import SelectMultiple, Dropdown, Output, HBox, VBox
from IPython.display import display, clear_output
from sklearn.datasets import fetch_california_housing, load_breast_cancer
from sklearn.linear_model import (LinearRegression, Ridge, Lasso,
                                   LogisticRegression)
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier,
                               GradientBoostingClassifier,
                               RandomForestRegressor,
                               GradientBoostingRegressor)
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import cross_validate, StratifiedKFold
np.random.seed(42)

try:
    import xgboost as xgb
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Describe** the performance of every supervised algorithm on a regression and a classification benchmark
- **Explain** why no single algorithm wins on every metric and every dataset
- **Interpret** a head-to-head comparison chart and use it to select the right algorithm for a specific problem

> **Tip:** Run the benchmark cells below before exploring the interactive widget. The benchmarks take 1-2 minutes to complete because they run cross-validation for every algorithm. Watch how the results vary — a strong AUC does not always correspond to high accuracy, especially on imbalanced datasets.

---
## How we got here

This notebook is the capstone of the supervised/ folder. Every algorithm benchmarked here has its own detailed notebook:

- **[01 Linear Regression](01_linear_regression.ipynb)** — [02 Multiple LR](02_multiple_linear_regression.ipynb) — [03 Polynomial](03_polynomial_regression.ipynb) — [04 Ridge/Lasso](04_ridge_lasso_regression.ipynb)**
- **[05 Logistic Regression](05_logistic_regression.ipynb)** — [06 KNN](06_k_nearest_neighbors.ipynb) — [07 Decision Trees](07_decision_trees.ipynb) — [08 Random Forests](08_random_forests.ipynb)**
- **[09 Gradient Boosting](09_gradient_boosting.ipynb)** — [10 XGBoost](10_xgboost.ipynb) — [11 SVM](11_support_vector_machines.ipynb) — [12 Naive Bayes](12_naive_bayes.ipynb)**

Also informed by: **[ml_concepts/13_interpretability_vs_complexity.ipynb](../ml_concepts/13_interpretability_vs_complexity.ipynb)** for the spectrum context.

---
## Why this matters for data science

No algorithm is universally best — this is the No Free Lunch theorem in practice. Every algorithm makes different assumptions, and those assumptions match some datasets better than others. The comparison in this notebook shows you the realistic range of performance differences and helps you build intuition for when each algorithm shines.

More importantly: looking at benchmark results alone is not enough. The right algorithm also depends on interpretability requirements, training time, prediction latency, and the cost of different types of errors. This notebook gives you all of those dimensions together.

---
## Regression benchmark: California Housing

Running all regression algorithms with 5-fold cross-validation on the California Housing dataset. Features are standardized where required. Metrics: R², RMSE, MAE.

In [ ]:
print("Loading California Housing dataset...")
housing = fetch_california_housing(as_frame=True)
X_h, y_h = housing.data, housing.target
scaler_h = StandardScaler()
X_h_scaled = scaler_h.fit_transform(X_h)
cv = 5
reg_scoring = ["r2", "neg_mean_squared_error", "neg_mean_absolute_error"]

# AveOccup and AveRooms have a few extreme outliers (AveOccup maxes out around
# 1,243 vs a mean of ~3) — squaring those for the polynomial model creates a
# handful of enormous leverage points that destabilize the fit, so that
# feature set alone is clipped to the 1st-99th percentile first.
X_h_clipped_scaled = StandardScaler().fit_transform(
    X_h.clip(lower=X_h.quantile(0.01), upper=X_h.quantile(0.99), axis=1)
)

reg_models = {
    "Linear Regression":    LinearRegression(),
    "Ridge":                Ridge(alpha=1.0),
    "Lasso":                Lasso(alpha=0.01, max_iter=5000),
    "Polynomial (deg=2)":   Pipeline([
                                ("poly", PolynomialFeatures(2, include_bias=False)),
                                ("sc",   StandardScaler()),
                                ("lr",   Ridge(alpha=1.0)),
                            ]),
    "Random Forest":        RandomForestRegressor(n_estimators=100, n_jobs=-1,
                                                   random_state=42),
    "Gradient Boosting":    GradientBoostingRegressor(n_estimators=100, learning_rate=0.1,
                                                       max_depth=3, random_state=42),
}

FEATURE_SETS = {
    "Linear Regression":  X_h_scaled,
    "Ridge":              X_h_scaled,
    "Lasso":              X_h_scaled,
    "Polynomial (deg=2)": X_h_clipped_scaled,
    "Random Forest":      X_h,
    "Gradient Boosting":  X_h,
}

reg_results = {}
for name, model in reg_models.items():
    X_use = FEATURE_SETS[name]
    scores = cross_validate(model, X_use, y_h, cv=cv, scoring=reg_scoring, n_jobs=-1)
    reg_results[name] = {
        "R²":   scores["test_r2"].mean(),
        "RMSE": np.sqrt(-scores["test_neg_mean_squared_error"]).mean(),
        "MAE":  -scores["test_neg_mean_absolute_error"].mean(),
    }
    print(f"  {name:25s}  R²={reg_results[name]['R²']:.4f}  "
          f"RMSE={reg_results[name]['RMSE']:.4f}  "
          f"MAE={reg_results[name]['MAE']:.4f}")
print("Regression benchmark complete.")

---
## Regression results

The chart below shows R² for all regression algorithms. Higher is better (max 1.0). RMSE and MAE are shown in the hover.

- **Notice:** Polynomial regression with Ridge can match or beat gradient boosting on well-behaved regression data
- **Notice:** Lasso R² may be lower than Ridge — it zeroes out features aggressively, which helps with irrelevant features but can hurt when all features are relevant
- **Notice:** The gap between the best and worst algorithm here is typically 0.10-0.20 R² — meaningful but not as dramatic as classification AUC gaps
- **Notice:** Polynomial regression here is evaluated on lightly outlier-clipped features, while the other models use the unclipped versions — a reasonable preprocessing choice, but worth knowing when comparing its R² directly against the others

In [ ]:
reg_names = list(reg_results.keys())
r2_vals   = [reg_results[n]["R²"]   for n in reg_names]
rmse_vals = [reg_results[n]["RMSE"] for n in reg_names]
mae_vals  = [reg_results[n]["MAE"]  for n in reg_names]

sorted_idx = sorted(range(len(r2_vals)), key=lambda i: r2_vals[i])
s_names = [reg_names[i] for i in sorted_idx]
s_r2    = [r2_vals[i]   for i in sorted_idx]
s_rmse  = [rmse_vals[i] for i in sorted_idx]
s_mae   = [mae_vals[i]  for i in sorted_idx]

fig = go.Figure(data=[
    go.Bar(y=s_names, x=s_r2, orientation='h',
           marker_color=PALETTE["primary"],
           text=[f"{v:.4f}" for v in s_r2], textposition="outside",
           hovertemplate="<b>%{y}</b><br>R²=%{x:.4f}<extra></extra>"),
], layout=base_layout(
    title="Regression Benchmark: 5-Fold CV R² — California Housing",
    xaxis_title="R²",
    yaxis_title="",
))
fig.update_layout(xaxis=dict(range=[0, 1.0]), height=380)
fig.show()

---
## Classification benchmark: Breast Cancer

Running all classification algorithms with 5-fold stratified cross-validation on the Breast Cancer dataset. Features are standardized where required. Metrics: Accuracy, Precision, Recall, F1, ROC-AUC.

In [ ]:
print("Loading Breast Cancer dataset...")
cancer = load_breast_cancer(as_frame=True)
X_c = cancer.data
# target_names = ['malignant', 'benign'], with 0 = malignant, 1 = benign.
# Flip so 1 = malignant (has cancer) — recall/precision default to treating
# class 1 as "positive," and the cancer-screening discussion later in this
# notebook is specifically about catching actual malignant cases.
y_c = (cancer.target == 0).astype(int)

scaler_c = StandardScaler()
X_c_scaled = scaler_c.fit_transform(X_c)
clf_scoring = ["accuracy", "precision", "recall", "f1", "roc_auc"]

clf_models = {
    "Logistic Regression": LogisticRegression(C=1.0, max_iter=1000, random_state=42),
    "KNN (k=7)":           KNeighborsClassifier(n_neighbors=7),
    "Decision Tree":       DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest":       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "Gradient Boosting":   GradientBoostingClassifier(n_estimators=100, learning_rate=0.1,
                                                       max_depth=3, random_state=42),
    "SVM (RBF)":           SVC(kernel='rbf', C=1.0, probability=True, random_state=42),
    "Naive Bayes":         GaussianNB(),
}

if XGB_AVAILABLE:
    clf_models["XGBoost"] = xgb.XGBClassifier(
        n_estimators=100, learning_rate=0.1, max_depth=3,
        use_label_encoder=False, eval_metric='logloss',
        random_state=42, verbosity=0,
    )

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
clf_results = {}
for name, model in clf_models.items():
    needs_scale = name in ("Logistic Regression", "KNN (k=7)", "SVM (RBF)")
    X_use = X_c_scaled if needs_scale else X_c.values

    scores = cross_validate(model, X_use, y_c, cv=skf, scoring=clf_scoring, n_jobs=-1)
    clf_results[name] = {
        "Accuracy":  scores["test_accuracy"].mean(),
        "Precision": scores["test_precision"].mean(),
        "Recall":    scores["test_recall"].mean(),
        "F1":        scores["test_f1"].mean(),
        "ROC-AUC":   scores["test_roc_auc"].mean(),
    }
    print(f"  {name:20s}  Acc={clf_results[name]['Accuracy']:.3f}  "
          f"F1={clf_results[name]['F1']:.3f}  AUC={clf_results[name]['ROC-AUC']:.3f}")
print("Classification benchmark complete.")

---
## Classification results

The chart below shows ROC-AUC for all classification algorithms. Higher is better (max 1.0). Hover to see all metrics.

- **Notice:** ROC-AUC and Accuracy can tell different stories — AUC is threshold-independent and generally more informative
- **Notice:** Naive Bayes often achieves competitive AUC despite its simplifying assumptions
- **Notice:** The best-performing algorithms (GBM, RF, XGBoost) have high complexity; the gap they open over logistic regression tells you how much nonlinearity exists in this dataset

In [ ]:
metric = "ROC-AUC"
clf_names = list(clf_results.keys())
metric_vals = [clf_results[n][metric] for n in clf_names]

sorted_idx = sorted(range(len(metric_vals)), key=lambda i: metric_vals[i])
s_names = [clf_names[i] for i in sorted_idx]
s_vals  = [metric_vals[i] for i in sorted_idx]
colors = [PALETTE["primary"] if v >= 0.80 else PALETTE["muted"] for v in s_vals]

fig = go.Figure(data=[
    go.Bar(y=s_names, x=s_vals, orientation='h',
           marker_color=colors,
           text=[f"{v:.4f}" for v in s_vals], textposition="outside"),
], layout=base_layout(
    title=f"Classification Benchmark: 5-Fold CV {metric} — Breast Cancer",
    xaxis_title=metric,
    yaxis_title="",
))
fig.update_layout(xaxis=dict(range=[0.5, 1.05]), height=400)
fig.show()

---
## All algorithms on the interpretability-complexity spectrum

See **[ml_concepts/13_interpretability_vs_complexity.ipynb](../ml_concepts/13_interpretability_vs_complexity.ipynb)** for the full discussion.

| Algorithm | Interpretability | Complexity | Typical use case |
|---|---|---|---|
| Linear / Multiple Regression | Very High | Very Low | Baseline, inference, regulated industries |
| Ridge / Lasso | Very High | Very Low | Many features, regularization needed |
| Logistic Regression | Very High | Very Low | Binary classification, probability outputs |
| Naive Bayes | Medium-High | Very Low | Text classification, streaming data |
| Decision Tree | High | Low-Medium | Auditable rules, quick exploratory model |
| KNN | Medium | Medium | Small datasets, recommendation systems |
| Polynomial Regression | Medium | Medium | Curved trends, few features |
| Random Forest | Medium | Medium-High | General-purpose, robust baseline |
| SVM (RBF) | Low-Medium | Medium-High | Small datasets, high-dimensional features |
| Gradient Boosting | Low | High | Maximum accuracy on tabular data |
| XGBoost | Low | Very High | Competitions, production tabular ML |
| Neural Network | Very Low | Very High | Images, text, audio, sequence data |

---
## How to choose your algorithm

The following criteria guide algorithm selection. Treat this as a starting point, not a rigid rule:

1. **Start with a linear model** (linear regression or logistic regression). It is fast, interpretable, and gives you a baseline.

2. **If linear performance is unsatisfactory**, check whether the residuals show nonlinear patterns. If yes, try polynomial features or tree-based models.

3. **If you have many features**, use regularized linear models (Ridge, Lasso) or tree-based models that do implicit feature selection.

4. **If interpretability is critical** (healthcare, finance, legal), stay in the top rows of the spectrum table above.

5. **If you need maximum accuracy and can tolerate a black box**, use Gradient Boosting or XGBoost. Tune carefully with cross-validation.

6. **If you are working with images, text, or audio**, skip the classical algorithms and start with neural networks.

7. **Always beat your baseline** before declaring any model production-ready.

In [ ]:
if "clf_results" not in globals() or "reg_results" not in globals():
    print("Run the regression and classification benchmark cells above first.")
else:
    REG_METRICS = ["R²", "RMSE", "MAE"]
    CLF_METRICS = ["ROC-AUC", "Accuracy", "Precision", "Recall", "F1"]

    dataset_dd = widgets.Dropdown(
        options=["Classification (Breast Cancer)", "Regression (California Housing)"],
        value="Classification (Breast Cancer)",
        description="Dataset:",
    )
    metric_dd = widgets.Dropdown(options=CLF_METRICS, value="ROC-AUC", description="Metric:")
    sort_dd = widgets.Dropdown(
        options=["By metric value", "By algorithm name"],
        value="By metric value",
        description="Sort:",
    )
    algo_select = widgets.SelectMultiple(
        options=list(clf_results.keys()), value=tuple(clf_results.keys()),
        description="Algorithms:", rows=8,
    )

    out = Output()

    def current_results():
        if dataset_dd.value.startswith("Regression"):
            return reg_results, REG_METRICS
        return clf_results, CLF_METRICS

    def on_dataset_change(change=None):
        results, metrics = current_results()
        algo_select.options = list(results.keys())
        algo_select.value = tuple(results.keys())
        metric_dd.options = metrics
        metric_dd.value = metrics[0]
        render()

    def render(change=None):
        results, metrics = current_results()
        metric = metric_dd.value if metric_dd.value in metrics else metrics[0]
        names = [a for a in algo_select.value if a in results]

        with out:
            clear_output(wait=True)
            if not names:
                print("Select at least one algorithm.")
                return

            vals = [results[n][metric] for n in names]
            if sort_dd.value == "By metric value":
                order = sorted(range(len(vals)), key=lambda i: vals[i])
            else:
                order = sorted(range(len(names)), key=lambda i: names[i], reverse=True)
            names = [names[i] for i in order]
            vals = [vals[i] for i in order]

            fig = go.Figure(data=[
                go.Bar(y=names, x=vals, orientation="h",
                       marker_color=PALETTE["primary"],
                       text=[f"{v:.4f}" for v in vals], textposition="outside"),
            ], layout=base_layout(
                title=f"{dataset_dd.value}: {metric} by algorithm",
                xaxis_title=metric,
                yaxis_title="",
            ))
            fig.update_layout(height=max(280, 40 * len(names)), showlegend=False)
            fig.show()

    dataset_dd.observe(on_dataset_change, names="value")
    algo_select.observe(render, names="value")
    metric_dd.observe(render, names="value")
    sort_dd.observe(render, names="value")

    display(VBox([
        HBox([dataset_dd, metric_dd, sort_dd]),
        algo_select,
        out,
    ]))
    render()

---
## Real-world example: Metric choice changes the winner

The same set of classifiers can have completely different rankings depending on which metric you use to evaluate them. This chart shows ROC-AUC alongside Recall for all algorithms — algorithms that rank highest on AUC are not always highest on Recall.

- **Notice:** Recall (sensitivity) measures how many actual positive cases the model catches; maximizing recall is critical when missing a positive is expensive (medical screening, fraud)
- **Notice:** An algorithm with high AUC but lower recall may be setting its threshold conservatively; you can always improve recall by lowering the threshold at the cost of precision
- **Notice:** The choice of evaluation metric is a business decision, not a modeling decision — define it before you start training

> **Discussion question:** A model for detecting early-stage cancer should prioritize which metric: precision (minimize false alarms) or recall (minimize missed cases)? What would convince you to accept 50 false alarms per true positive?

### Evaluation metric guide

| Metric | Definition | Maximize when |
|---|---|---|
| R² | Fraction of variance explained (regression) | You need to explain prediction quality simply |
| RMSE | Root mean squared error — penalizes large errors | Large errors are very costly |
| MAE | Mean absolute error — robust to outliers | Outliers are common and should not dominate |
| Accuracy | Fraction of correct predictions | Classes are balanced and all errors equal |
| Precision | True positives / predicted positives | False positives are very costly (spam filter) |
| Recall | True positives / actual positives | False negatives are very costly (cancer screening) |
| F1 | Harmonic mean of precision and recall | Both precision and recall matter equally |
| ROC-AUC | Area under the ROC curve | Comparing classifiers regardless of threshold |

In [ ]:
if clf_results:
    names  = list(clf_results.keys())
    aucs   = [clf_results[n]["ROC-AUC"] for n in names]
    recalls = [clf_results[n]["Recall"] for n in names]

    fig = go.Figure(data=[
        go.Bar(name="ROC-AUC", x=names, y=aucs,
               marker_color=PALETTE["primary"], opacity=0.85,
               text=[f"{v:.3f}" for v in aucs], textposition="outside"),
        go.Bar(name="Recall", x=names, y=recalls,
               marker_color=PALETTE["secondary"], opacity=0.85,
               text=[f"{v:.3f}" for v in recalls], textposition="outside"),
    ], layout=base_layout(
        title="Metric Matters: ROC-AUC vs Recall — Same Algorithms, Different Story",
        xaxis_title="Algorithm",
        yaxis_title="Score",
    ))
    fig.update_layout(barmode="group", yaxis=dict(range=[0, 1.15]),
                      xaxis=dict(tickangle=-25), height=420)
    fig.show()
else:
    print("Run the classification benchmark cell first.")

> **No algorithm wins on every metric and every dataset — the right choice depends on your data, your interpretability constraints, and the real-world cost of each type of error.**

---
*You now know every major supervised learning algorithm. Next we explore what happens when there are no labels.*

---
*Next up: unsupervised/ — clustering, dimensionality reduction, and anomaly detection without labeled data*